In [106]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from catboost import CatBoostRegressor
import xgboost as xgb
# 1. Chargement des données (Utilise tes noms de fichiers sauvegardés)
X_train = pd.read_csv("../features/global_features/X_train_filtered.csv")
y_train = pd.read_csv("../features/global_features/y_train.csv")
X_test = pd.read_csv("../features/global_features/X_test_filtered.csv")

In [107]:
# 2. Préparation
y_train = y_train.values.flatten()
test_ids = X_test['video_id']
X = X_train.drop(columns=['video_id'], errors='ignore')
X_test_final = X_test.drop(columns=['video_id'], errors='ignore')

# 3. Configuration du K-Fold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Listes pour stocker les scores de validation et les prédictions finales
oof_preds = np.zeros(len(X)) # Out-of-fold predictions
#test_preds = np.zeros(len(X_test_final))
cv_scores = []

X_train = pd.DataFrame(X_train)
y_train = pd.DataFrame(y_train)
X_test = pd.DataFrame(X_test)
# On isole l'ID pour la soumission finale (très important !)
test_ids = X_test['video_id'].copy()

# On définit les features en supprimant video_id
# errors='ignore' permet de ne pas planter si la colonne est déjà absente
X_train = X_train.drop(columns=['video_id'], errors='ignore')
X_test = X_test.drop(columns=['video_id'], errors='ignore')

# On s'assure que y_train est un array 1D pour les calculs de metrics
# y_train doit être la colonne 'score' uniquement
y_train_values = y_train.values.flatten()

In [108]:
# =========================
# 3) LightGBM params
# =========================
lgb_params2 = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.015,     # Plus lent pour ne pas rater l'optimum
    "num_leaves": 15,          # Très bas pour éviter l'overfitting
    "max_depth": 4,            # On force des arbres courts (plus robustes)
    "min_child_samples": 40,   # On force chaque feuille à avoir au moins 40 vidéos
    "feature_fraction": 0.5,   # On ne prend que 50% des colonnes par arbre
    "reg_alpha": 1.0,          # L1 plus fort
    "reg_lambda": 5.0,         # L2 plus fort
    "verbose": -1,
    "random_state": 42
}
lgb_params ={
    "objective": "regression",
    "metric": "rmse",
 'learning_rate': 0.024489296516914404, 
 'num_leaves': 98, 
 'max_depth': 9, 
 'min_child_samples': 51, 
 'feature_fraction': 0.6918923726742648, 
 'bagging_fraction': 0.8307449535209684, 
 'bagging_freq': 6, 
 'reg_alpha': 1.0725069061880557, 
 'reg_lambda': 0.0757098806219079,
 "random_state": 42}
lgb_params_4 = {'learning_rate': 0.04689432875992573, 
              'num_leaves': 81, 
              'max_depth': 6, 
              'min_child_samples': 58, 
              'feature_fraction': 0.4667450301146786, 
              'bagging_fraction': 0.8300706683532761, 
              'bagging_freq': 2, 
              'reg_alpha': 0.04401878759910092, 
              'reg_lambda': 0.08502967604602517,
              "random_state": 42,
              "objective": "regression",
              "metric": "rmse"
              }
# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds   = np.zeros(len(X_train))
test_preds  = np.zeros(len(X_test))
feature_cols = X_train.columns # Maintenant sans video_id
feature_imp = np.zeros(len(feature_cols))
print(feature_cols)
print(f"\n{'='*50}")
print(f"KFold CV — 5 folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_values)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_values[train_idx], y_train_values[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test) / 5
    feature_imp        += model.feature_importances_ / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

# Utilisation de y_train_values pour le calcul final
oof_rmse = np.sqrt(mean_squared_error(y_train_values, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")

Index(['c_pc1', 'c_pc2', 'c_pc3', 'c_pc4', 'c_pc5', 'c_pc7', 'c_pc17',
       'text_len', 'nb_hashtags', 'up_uploader_kitzbuehel_official',
       'up_uploader_megeve_officiel', 'up_uploader_stanton_arlberg_official',
       'up_uploader_val_thorens', 'v_pc1', 'v_pc5', 'v_pc7'],
      dtype='object')

KFold CV — 5 folds
[200]	valid_0's rmse: 1.20205
  Fold 1 | Best iter:  156 | RMSE: 1.1981
[200]	valid_0's rmse: 1.31708
  Fold 2 | Best iter:  164 | RMSE: 1.3096
[200]	valid_0's rmse: 1.25914
  Fold 3 | Best iter:  126 | RMSE: 1.2473
[200]	valid_0's rmse: 1.21607
  Fold 4 | Best iter:  227 | RMSE: 1.2139
[200]	valid_0's rmse: 1.29451
  Fold 5 | Best iter:  219 | RMSE: 1.2937

OOF RMSE global : 1.2533


In [93]:
# 1. Paramètres optimisés pour ton petit dataset (1348 lignes)
cb_params2 = {
    'iterations': 2000,          # Nombre maximum d'arbres
    'learning_rate': 0.02,       # Pas d'apprentissage lent pour la précision
    'depth': 6,                  # Profondeur modérée pour éviter d'apprendre le bruit
    'l2_leaf_reg': 5,            # Régularisation L2 forte
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': 200,              # Affiche le score tous les 200 arbres
    'early_stopping_rounds': 100 # Stop si le RMSE ne baisse plus pendant 100 tours
}
cb_params = {
    'learning_rate': 0.04308672251201548, 
    'depth': 5, 'l2_leaf_reg': 1.7506704107126339, 
    'bootstrap_type': 'Bernoulli', 
    'random_strength': 0.696136089192006, 
    'subsample': 0.5905605688531387}

# 2. Préparation des tableaux de résultats
cb_oof_preds = np.zeros(len(X_train))
cb_test_preds = np.zeros(len(X_test))

print(f"\n{'='*50}")
print(f"CatBoost KFold CV — 5 folds")
print(f"{'='*50}")

# 3. Boucle K-Fold
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_values)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_values[train_idx], y_train_values[val_idx]

    model_cb = CatBoostRegressor(**cb_params)
    
    model_cb.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True,
        plot=False # Met à True si tu es sur Jupyter pour voir la courbe en temps réel
    )

    cb_oof_preds[val_idx] = model_cb.predict(X_val)
    cb_test_preds         += model_cb.predict(X_test) / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, cb_oof_preds[val_idx]))
    print(f"  Fold {fold+1} | RMSE: {fold_rmse:.4f}")

cb_oof_rmse = np.sqrt(mean_squared_error(y_train_values, cb_oof_preds))
print(f"\nOOF CatBoost RMSE global : {cb_oof_rmse:.4f}")


CatBoost KFold CV — 5 folds
0:	learn: 1.6978311	test: 1.5699976	best: 1.5699976 (0)	total: 3.08ms	remaining: 3.08s
1:	learn: 1.6853903	test: 1.5590060	best: 1.5590060 (1)	total: 3.84ms	remaining: 1.92s
2:	learn: 1.6744821	test: 1.5487050	best: 1.5487050 (2)	total: 4.78ms	remaining: 1.59s
3:	learn: 1.6612024	test: 1.5369157	best: 1.5369157 (3)	total: 5.59ms	remaining: 1.39s
4:	learn: 1.6511383	test: 1.5278941	best: 1.5278941 (4)	total: 6.4ms	remaining: 1.27s
5:	learn: 1.6401317	test: 1.5175366	best: 1.5175366 (5)	total: 7.28ms	remaining: 1.21s
6:	learn: 1.6304903	test: 1.5105898	best: 1.5105898 (6)	total: 9.61ms	remaining: 1.36s
7:	learn: 1.6208881	test: 1.5014929	best: 1.5014929 (7)	total: 11.6ms	remaining: 1.44s
8:	learn: 1.6103578	test: 1.4906630	best: 1.4906630 (8)	total: 14.7ms	remaining: 1.62s
9:	learn: 1.6002275	test: 1.4803520	best: 1.4803520 (9)	total: 17.3ms	remaining: 1.71s
10:	learn: 1.5931863	test: 1.4756553	best: 1.4756553 (10)	total: 21.8ms	remaining: 1.96s
11:	learn: 1.

In [86]:
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from catboost import CatBoostRegressor

# On réutilise tes données déjà préparées
X = X_train.drop(columns=['video_id'], errors='ignore')
y = y_train_values  # L'array 1D créé dans ta cellule 41

def objective_lgb(trial):
    # Espace de recherche pour LightGBM
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 42,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model = lgb.LGBMRegressor(n_estimators=1000, **params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        
        preds = model.predict(X_val)
        scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        
    return np.mean(scores)

def objective_cb(trial):
    # Espace de recherche pour CatBoost
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli"]),
        "random_strength": trial.suggest_float("random_strength", 1e-8, 10.0, log=True),
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
        "random_seed": 42
    }
    
    if params["bootstrap_type"] == "Bernoulli":
        params["subsample"] = trial.suggest_float("subsample", 0.4, 1.0)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
        
        preds = model.predict(X_val)
        scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        
    return np.mean(scores)

# --- Exécution de l'optimisation ---
print("Optimisation LightGBM...")
study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(objective_lgb, n_trials=50) # Augmente à 100 pour de meilleurs résultats
print("\nMeilleurs paramètres LGBM:", study_lgb.best_params)

[I 2026-03-03 23:36:46,512] A new study created in memory with name: no-name-cb7b7a1b-0196-4496-bb25-9484c7ee8395


Optimisation LightGBM...


[I 2026-03-03 23:36:47,393] Trial 0 finished with value: 1.240184586857201 and parameters: {'learning_rate': 0.032561382188123045, 'num_leaves': 56, 'max_depth': 5, 'min_child_samples': 82, 'feature_fraction': 0.9666868361659954, 'bagging_fraction': 0.6270777894926389, 'bagging_freq': 6, 'reg_alpha': 0.005237409313912274, 'reg_lambda': 1.1060764201770525}. Best is trial 0 with value: 1.240184586857201.
[I 2026-03-03 23:36:48,443] Trial 1 finished with value: 1.3044263685500872 and parameters: {'learning_rate': 0.029434502415181613, 'num_leaves': 49, 'max_depth': 5, 'min_child_samples': 78, 'feature_fraction': 0.8924355228543472, 'bagging_fraction': 0.497375660298923, 'bagging_freq': 5, 'reg_alpha': 0.033402280872303666, 'reg_lambda': 0.012145260649532733}. Best is trial 0 with value: 1.240184586857201.
[I 2026-03-03 23:36:49,435] Trial 2 finished with value: 1.22671584167747 and parameters: {'learning_rate': 0.04349213046229746, 'num_leaves': 19, 'max_depth': 10, 'min_child_samples': 1


Meilleurs paramètres LGBM: {'learning_rate': 0.04689432875992573, 'num_leaves': 81, 'max_depth': 6, 'min_child_samples': 58, 'feature_fraction': 0.4667450301146786, 'bagging_fraction': 0.8300706683532761, 'bagging_freq': 2, 'reg_alpha': 0.04401878759910092, 'reg_lambda': 0.08502967604602517}


In [8]:
print("Optimisation CatBoost...")
study_cb = optuna.create_study(direction="minimize")
study_cb.optimize(objective_cb, n_trials=30) # CatBoost est plus lent, on fait moins de trials
print("Meilleurs paramètres CatBoost:", study_cb.best_params)

[I 2026-03-03 22:35:53,125] A new study created in memory with name: no-name-d5c963e1-d3db-46e3-8c45-7230a96fe114


Optimisation CatBoost...


[I 2026-03-03 22:37:17,598] Trial 0 finished with value: 1.2604026028172415 and parameters: {'learning_rate': 0.0069055279780114165, 'depth': 8, 'l2_leaf_reg': 5.318454308069668, 'bootstrap_type': 'Bayesian', 'random_strength': 2.1444617097745225e-05}. Best is trial 0 with value: 1.2604026028172415.
[I 2026-03-03 22:41:52,765] Trial 1 finished with value: 1.294308232615461 and parameters: {'learning_rate': 0.01251301754185757, 'depth': 10, 'l2_leaf_reg': 9.777111241534923, 'bootstrap_type': 'Bayesian', 'random_strength': 0.0020680417409777167}. Best is trial 0 with value: 1.2604026028172415.
[I 2026-03-03 22:42:26,408] Trial 2 finished with value: 1.229991044304377 and parameters: {'learning_rate': 0.021139026957624645, 'depth': 7, 'l2_leaf_reg': 1.1903666968875106, 'bootstrap_type': 'Bernoulli', 'random_strength': 0.1207461880870965, 'subsample': 0.8627173137060549}. Best is trial 2 with value: 1.229991044304377.
[I 2026-03-03 22:43:56,976] Trial 3 finished with value: 1.2451127882373

Meilleurs paramètres CatBoost: {'learning_rate': 0.04308672251201548, 'depth': 5, 'l2_leaf_reg': 1.7506704107126339, 'bootstrap_type': 'Bernoulli', 'random_strength': 0.696136089192006, 'subsample': 0.5905605688531387}


In [89]:
final_preds = (0.5 * cb_test_preds) + \
                       (0.5 * test_preds)

# 5. Sauvegarde de la nouvelle soumission
submission = pd.DataFrame({
    'ID': test_ids,
    'popularity': final_preds
})
submission.to_csv("submission_filtered_seuil.csv", index=False)